# HW3 — Smoke test / environment bring-up

**Σκοπός:** να επιβεβαιώσουμε ότι το pipeline τρέχει end-to-end στο Kaggle πριν χτίσουμε
το κανονικό experiment harness. Εδώ **δεν** κάνουμε πειράματα — μόνο smoke test.

Τι θέλουμε να μάθουμε:
1. ότι το environment install δουλεύει (Qwen3.5 + latest transformers),
2. τι δομή έχουν τα δεδομένα του Kaggle competition (columns, labels, μέγεθος test set),
3. πώς συμπεριφέρεται το Qwen "thinking mode" (πόσα tokens / χρόνο καίει),
4. πόσο χρόνο θέλει ανά sample το μικρό μοντέλο (Qwen3.5-0.8B).

## Πριν τρέξεις — checklist

- **Settings → Accelerator → GPU** (T4 x2 ή ό,τι δίνει το Kaggle).
- **Settings → Internet → On** (χρειάζεται για το pip install και το dataset).
- **Add Input →** πρόσθεσε το dataset του competition του Assignment 3.
- Αν μετά το install cell σου ζητήσει restart: κάνε **Restart** και ξανατρέξε από την αρχή.

Όταν τελειώσει, στείλε μου πίσω **όλο** το output (δες το τελευταίο cell).

In [1]:
!pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 10.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 76.7 MB/s eta 0:00:00:00:01


## 1. Environment check

In [2]:
import time, os, glob, gc
import numpy as np
import pandas as pd
import torch
import transformers

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

transformers: 5.8.0.dev0
torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU count: 2


## 2. Δεδομένα του competition — τι αρχεία μας έδωσαν

Εδώ απλώς κοιτάμε τι υπάρχει στο `/kaggle/input`. Αυτό μας λέει το schema (columns,
ονόματα labels, μέγεθος test set) που θα χρειαστούμε στη Phase 2 για το πραγματικό submission.

In [3]:
# Ολα τα αρχεια που ειναι attached στο notebook.
found = False
for root, _, files in os.walk("/kaggle/input"):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(p, f"({os.path.getsize(p) / 1024:.0f} KB)")
        found = True
if not found:
    print("Δεν βρεθηκε τιποτα στο /kaggle/input — μηπως δεν εγινε Add Input το competition dataset;")

/kaggle/input/competitions/ys-19-2025-2026-assignment-3/sample_solution.csv (5 KB)


In [4]:
# Φορτωσε καθε CSV που θα βρει και τυπωσε shape / columns / head.
csv_paths = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print("CSV files:", csv_paths, "\n")

tables = {}
for p in csv_paths:
    name = os.path.basename(p)
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f"[skip] {name}: {e}")
        continue
    tables[name] = df
    print("=" * 70)
    print(f"{name}  | shape: {df.shape}")
    print("columns:", list(df.columns))
    print(df.head(3).to_string())
    print()

CSV files: ['/kaggle/input/competitions/ys-19-2025-2026-assignment-3/sample_solution.csv'] 

sample_solution.csv  | shape: (308, 2)
columns: ['Id', 'Predicted']
   Id        Predicted
0   0       Ambivalent
1   1       Ambivalent
2   2  Clear Non-Reply



In [5]:
# Προσπαθησε να βρεις το label column και τυπωσε την κατανομη κλασεων.
CLARITY = {"Clear Reply", "Ambivalent", "Clear Non-Reply"}
for name, df in tables.items():
    for col in df.columns:
        uniq = set(str(v) for v in df[col].dropna().unique())
        if uniq and uniq <= CLARITY:
            print(f"{name} -> label column '{col}':")
            print(df[col].value_counts(), "\n")

sample_solution.csv -> label column 'Predicted':
Predicted
Clear Non-Reply    114
Clear Reply         99
Ambivalent          95
Name: count, dtype: int64 



## 3. Φόρτωση μοντέλου — Qwen3.5-0.8B

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-0.8B"

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map="auto")
print(f"model + tokenizer loaded in {time.time() - t0:.1f}s")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

model + tokenizer loaded in 11.7s


## 4. Μικρό δείγμα για το smoke test

Για το smoke χρησιμοποιούμε το HF dataset του tutorial (`ailsntua/QEvasion`) γιατί έχει
γνωστό, σταθερό schema. Το schema του competition (ενότητα 2) θα το χρησιμοποιήσουμε από
τη Phase 2 και μετά για το κανονικό submission.

In [7]:
from datasets import load_dataset

N_SMOKE = 32
ds = load_dataset("ailsntua/QEvasion", split=f"train[:{N_SMOKE}]")
print("fields:", ds.column_names)
print("\nπαραδειγμα [0]:")
for k, v in ds[0].items():
    print(f"  {k}: {str(v)[:200]}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/308 [00:00<?, ? examples/s]

fields: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label']

παραδειγμα [0]:
  title: The President's News Conference in Hanoi, Vietnam
  date: September 10, 2023
  president: Joseph R. Biden
  url: https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-hanoi-vietnam-0
  question_order: 1
  interview_question: Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about
  interview_answer: Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in t

## 5. Zero-shot prompt + thinking-mode probe

In [8]:
LABELS = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

system_prompt = """You are a political response classifier.
Given a question and its corresponding answer, classify the answer into one of:
- Clear Reply
- Ambivalent
- Clear Non-Reply
Respond with the label only."""

def build_user_prompt(ex):
    return f'Question: {ex["question"]}\nAnswer: {ex["interview_answer"]}'

print(system_prompt)
print("\n--- user prompt example ---\n")
print(build_user_prompt(ds[0])[:600])

You are a political response classifier.
Given a question and its corresponding answer, classify the answer into one of:
- Clear Reply
- Ambivalent
- Clear Non-Reply
Respond with the label only.

--- user prompt example ---

Question: How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?
Answer: Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was about


### 5α. Ένα παράδειγμα με thinking mode στο default

Βάζουμε σκόπιμα μεγάλο `max_new_tokens=512` για να **δούμε** αν το μοντέλο μπαίνει σε
μεγάλο thinking trace.

In [9]:
ex = ds[0]
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": build_user_prompt(ex)},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
dt = time.time() - t0
gen = out[0][inputs["input_ids"].shape[1]:]
print(f"thinking=default | new tokens: {len(gen)} | time: {dt:.1f}s | gold: {ex['clarity_label']}")
print("\nRAW OUTPUT:")
print(repr(tokenizer.decode(gen, skip_special_tokens=True)))

thinking=default | new tokens: 7 | time: 1.9s | gold: Clear Reply

RAW OUTPUT:
'Clear Non-Reply\n'


### 5β. Το ίδιο παράδειγμα με `enable_thinking=False`

Δοκιμάζουμε αν το chat template του Qwen3.5 δέχεται `enable_thinking=False` και τι αλλάζει
σε tokens / χρόνο.

In [10]:
try:
    text_nt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text_nt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    dt = time.time() - t0
    gen = out[0][inputs["input_ids"].shape[1]:]
    print(f"enable_thinking=False | new tokens: {len(gen)} | time: {dt:.1f}s")
    print("\nRAW OUTPUT:")
    print(repr(tokenizer.decode(gen, skip_special_tokens=True)))
except Exception as e:
    print("enable_thinking=False δεν εγινε δεκτο απο το chat template:")
    print(repr(e))

enable_thinking=False | new tokens: 7 | time: 0.7s

RAW OUTPUT:
'Clear Non-Reply\n'


## 6. Batched inference + timing στο μικρό δείγμα

Τρέχουμε και τα 32 samples σε batches για να μετρήσουμε ρεαλιστικό χρόνο ανά sample.
Αν βγει **OOM**, ρίξε το `BATCH_SIZE` σε 4 ή 1.

In [11]:
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

BATCH_SIZE = 8
MAX_NEW_TOKENS = 512

def run_batch(dataset, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    prompts = []
    for ex in dataset:
        msgs = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": build_user_prompt(ex)}]
        prompts.append(tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True))
    outs = []
    t0 = time.time()
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]
            outs.append(tokenizer.decode(new, skip_special_tokens=True))
    return outs, time.time() - t0

generations, elapsed = run_batch(ds)
print(f"{len(generations)} samples σε {elapsed:.1f}s  ->  {elapsed / len(generations):.2f}s/sample")

32 samples σε 15.5s  ->  0.49s/sample


## 7. Parsing + γρήγορα metrics

In [12]:
def parse_label(text):
    t = text.strip()
    # Αν υπαρχει thinking trace, κρατα μονο ο,τι ειναι μετα το </think>.
    if "</think>" in t:
        t = t.split("</think>")[-1].strip()
    # Ταξινομηση κατα μηκος ωστε το "Clear Reply" να μην ταιριαξει μεσα στο "Clear Non-Reply".
    for lab in sorted(LABELS, key=len, reverse=True):
        if lab.lower() in t.lower():
            return lab
    return "Invalid"

preds = [parse_label(g) for g in generations]
gold = [ds[i]["clarity_label"] for i in range(len(ds))]

valid = sum(p != "Invalid" for p in preds)
acc = np.mean([p == g for p, g in zip(preds, gold)])
print(f"valid outputs : {valid}/{len(preds)}")
print(f"accuracy      : {acc:.3f}  (smoke, N={len(preds)})")
print(f"pred dist     : {pd.Series(preds).value_counts().to_dict()}")
print(f"gold dist     : {pd.Series(gold).value_counts().to_dict()}")

print("\n--- πρωτα 8 παραδειγματα ---")
for i in range(min(8, len(preds))):
    print(f"[{i}] gold={gold[i]:<16} pred={preds[i]:<16} raw={repr(generations[i][:160])}")

valid outputs : 32/32
accuracy      : 0.188  (smoke, N=32)
pred dist     : {'Clear Non-Reply': 24, 'Ambivalent': 6, 'Clear Reply': 2}
gold dist     : {'Ambivalent': 20, 'Clear Reply': 9, 'Clear Non-Reply': 3}

--- πρωτα 8 παραδειγματα ---
[0] gold=Clear Reply      pred=Clear Non-Reply  raw='Clear Non-Reply\n'
[1] gold=Ambivalent       pred=Clear Non-Reply  raw='Clear Non-Reply\n'
[2] gold=Ambivalent       pred=Clear Non-Reply  raw='Clear Non-Reply\n'
[3] gold=Ambivalent       pred=Clear Non-Reply  raw='Clear Non-Reply\n'
[4] gold=Clear Reply      pred=Clear Non-Reply  raw='Clear Non-Reply\n'
[5] gold=Ambivalent       pred=Clear Non-Reply  raw='Clear Non-Reply\n'
[6] gold=Ambivalent       pred=Clear Non-Reply  raw='Clear Non-Reply\n'
[7] gold=Ambivalent       pred=Clear Non-Reply  raw='Clear Non-Reply\n'


## 8. Τι να μου στείλεις πίσω

Αντέγραψε και στείλε μου **όλο** το output που τύπωσαν τα cells, ειδικά:

- **Ενότητα 1** — versions + GPU
- **Ενότητα 2** — λίστα αρχείων, columns/head των CSV, κατανομή labels του competition
- **Ενότητα 5** — raw output με thinking on/off (tokens + χρόνος και στις δύο περιπτώσεις)
- **Ενότητα 6** — το `s/sample` του batched run
- **Ενότητα 7** — valid count, accuracy, distributions, raw παραδείγματα

Με αυτά: (1) κλείνω το dataset schema, (2) ρυθμίζω thinking + `max_new_tokens`,
(3) αποφασίζουμε Kaggle-only vs Colab Pro, (4) χτίζω το harness της Phase 2.